In [6]:

from scipy.ndimage import uniform_filter as uf
import pandas as pd
import os
import glob



import numpy as np
from numpy.fft import fft2,ifft2,fftshift
import matplotlib
from matplotlib import pyplot as plt 
from matplotlib import cm
import scipy
from scipy.ndimage import gaussian_filter1d as gf1d
from scipy.ndimage import gaussian_filter as gf
from scipy.ndimage import uniform_filter as uf

import sys
sys.path.append(("F:\\python for ddm and sia to be saved on desktop\\PyDDM")) #must point to the PyDDM folder

import ddm_analysis_and_fitting as ddm

sys.path.append("F:\\python for ddm and sia to be saved on desktop\\ddmanalysis")

import tiff_file
import ddm_clean as ddm

#import xlsxwriter
import pandas as pd   
import math



# Define the directory where your TIFF files are located
data_dir = ("F:\\sia-exp1,2-control,blebb,cytd-analyzed tobe plotted\\wt-vn-sia-exp1-ctd.blebb,controlzstack-analyzed to be plotted\\control-wt\\wt-smol region-300by300")

# Define the function to process a single tiff image




def im_corr(image, filter=False, filtersize=512):
    if filter:
        image = image * 1.0 - uf(image, filtersize)
    image = 1.0 * image - image.mean()
    image = image / image.std()
    corr_im = np.real(fftshift(ifft2(fft2(image) * np.conj(fft2(image))))) / (image.shape[0] * image.shape[1])
    
    center = (corr_im.shape[0] // 2, corr_im.shape[1] // 2)
    g0 = corr_im[center[0], center[1]]
    
    print(f"g(0) from central pixel = {g0}")
    
    rav_corr = ddm.newRadav(corr_im)  # This averages into radial profile
    return corr_im, rav_corr


#def im_corr(image, filter=False, filtersize=512):
    #if filter:
    #image = 1.0 * image - image.mean()
    #image = image / image.std()
    #corr_im = np.real(fftshift(ifft2(fft2(image) * np.conj(fft2(image))))) / (image.shape[0] * image.shape[1])
    #rav_corr = ddm.newRadav(corr_im)  # Assuming newRadav is part of the ddm module
    #return corr_im, rav_corr

# Define the function to process a single tiff file and save the results to Excel
def process_tiff_file(file_path, filtersize=256, m2p_ratio=0.07, num=1):
    # Load the image
    im = tiff_file.imread(file_path)
    
    # Reshape for processing
    im_main = im.reshape(1, im.shape[0], im.shape[1])
    
    # Initialize arrays for correlation and radial averages
    corr_im = np.zeros_like(im_main[:,:,:])
    corr_ravs = np.zeros((im_main.shape[0], int(im_main.shape[1] / 2)))

    # Process each frame in the tiff file
    for i in range(im_main.shape[0]):
        corr_im[i], temp = im_corr(im_main[i, :, :], filter=False, filtersize=filtersize)
        corr_ravs[i] = temp[:corr_ravs.shape[1]]
    
    # Downsample the correlation results if necessary
    corr_ravs_pick = corr_ravs.shape[0] / num
    gr_p = np.zeros((int(corr_ravs_pick), corr_ravs.shape[1]))

    n = 0
    for i in range(int(corr_ravs_pick)):
        gr_p[i, :] = corr_ravs[n, :]
        n += num

    # Compute the averages
    gravgse = np.zeros((gr_p.shape[1], 3))
    xvalues = np.arange(corr_ravs.shape[1]) * m2p_ratio
    x_vals = np.zeros((len(xvalues), 1))

    for i in range(len(xvalues)):
        x_vals[i, 0] = xvalues[i]

    for i in range(gr_p.shape[1]):
        gravgse[i, 0] = x_vals[i, 0]
        gravgse[i, 1] = np.mean(gr_p[:, i])

    # Save the data to Excel
    output_filename = os.path.join(data_dir, os.path.splitext(os.path.basename(file_path))[0] + '.xlsx')
    df_tot = pd.DataFrame(gr_p[:, :])
    dfavgse = pd.DataFrame(gravgse[:, :])

    with pd.ExcelWriter(output_filename, mode='w', engine='openpyxl') as writer:
       dfavgse.to_excel(writer, sheet_name="vn-A-2-crop_wfilter-avgstd", header=False, index=False)

    print(f"Processed and saved {output_filename}")
    return df_tot, dfavgse

# Define the function to process all tiff files in a folder
def process_tiff_folder(folder_path, filtersize=256):
    # Get all .tif files in the folder
    tiff_files = glob.glob(os.path.join(folder_path, '*.tif'))

    for tiff_file_path in tiff_files:
        print(f"Processing {tiff_file_path}...")
        process_tiff_file(tiff_file_path, filtersize=filtersize)

# Run the function for the entire folder
process_tiff_folder(data_dir)


Processing F:\sia-exp1,2-control,blebb,cytd-analyzed tobe plotted\wt-vn-sia-exp1-ctd.blebb,controlzstack-analyzed to be plotted\control-wt\wt-smol region-300by300\wt.tif...
g(0) from central pixel = 1.0
Processed and saved F:\sia-exp1,2-control,blebb,cytd-analyzed tobe plotted\wt-vn-sia-exp1-ctd.blebb,controlzstack-analyzed to be plotted\control-wt\wt-smol region-300by300\wt.xlsx
Processing F:\sia-exp1,2-control,blebb,cytd-analyzed tobe plotted\wt-vn-sia-exp1-ctd.blebb,controlzstack-analyzed to be plotted\control-wt\wt-smol region-300by300\wt001.tif...
g(0) from central pixel = 1.0000000000000002
Processed and saved F:\sia-exp1,2-control,blebb,cytd-analyzed tobe plotted\wt-vn-sia-exp1-ctd.blebb,controlzstack-analyzed to be plotted\control-wt\wt-smol region-300by300\wt001.xlsx
Processing F:\sia-exp1,2-control,blebb,cytd-analyzed tobe plotted\wt-vn-sia-exp1-ctd.blebb,controlzstack-analyzed to be plotted\control-wt\wt-smol region-300by300\wt001-1.tif...
g(0) from central pixel = 1.0
Proce

In [2]:
from scipy.ndimage import uniform_filter as uf
import pandas as pd
import os
import glob
import numpy as np
from numpy.fft import fft2, ifft2, fftshift
import matplotlib
from matplotlib import pyplot as plt 
from matplotlib import cm
import scipy
from scipy.ndimage import gaussian_filter1d as gf1d
from scipy.ndimage import gaussian_filter as gf
import sys

# Paths
sys.path.append("F:\\python for ddm and sia to be saved on desktop\\PyDDM")
import ddm_analysis_and_fitting as ddm
sys.path.append("F:\\python for ddm and sia to be saved on desktop\\ddmanalysis")
import tiff_file
import ddm_clean as ddm

# Data directory
data_dir = ("F:\\sia-exp1,2-control,blebb,cytd-analyzed tobe plotted\\wt-vn-sia-exp1-ctd.blebb,controlzstack-analyzed to be plotted\\blebb-vn-sia\\vn-smol region-300by 300")


# Function to calculate correlation image and radial average
def im_corr(image, filter=False, filtersize=512):
    if filter:
        image = image * 1.0 - uf(image, filtersize)
    image = 1.0 * image - image.mean()
    image = image / image.std()
    corr_im = np.real(fftshift(ifft2(fft2(image) * np.conj(fft2(image))))) / (image.shape[0] * image.shape[1])

    # Get true g(0) from center pixel
    center = (corr_im.shape[0] // 2, corr_im.shape[1] // 2)
    g0 = corr_im[center[0], center[1]]
    print(f"g(0) from central pixel = {g0}")

    rav_corr = ddm.newRadav(corr_im)  # Radial average
    return corr_im, rav_corr, g0

# Function to process a single tiff file
def process_tiff_file(file_path, filtersize=256, m2p_ratio=0.07, num=1):
    im = tiff_file.imread(file_path)
    im_main = im.reshape(1, im.shape[0], im.shape[1])

    corr_im = np.zeros_like(im_main)
    corr_ravs = np.zeros((im_main.shape[0], int(im_main.shape[1] / 2)))
    g0_value = None

    # Process each frame
    for i in range(im_main.shape[0]):
        corr_im[i], temp, g0 = im_corr(im_main[i, :, :], filter=False, filtersize=filtersize)
        corr_ravs[i] = temp[:corr_ravs.shape[1]]
        if g0_value is None:
            g0_value = g0  # Save g(0) from first frame (or you can average over frames if you want)

    # Downsample
    corr_ravs_pick = corr_ravs.shape[0] / num
    gr_p = np.zeros((int(corr_ravs_pick), corr_ravs.shape[1]))

    n = 0
    for i in range(int(corr_ravs_pick)):
        gr_p[i, :] = corr_ravs[n, :]
        n += num

    # Compute averages
    gravgse = np.zeros((gr_p.shape[1], 3))
    xvalues = np.arange(corr_ravs.shape[1]) * m2p_ratio
    x_vals = np.zeros((len(xvalues), 1))

    for i in range(len(xvalues)):
        x_vals[i, 0] = xvalues[i]

    for i in range(gr_p.shape[1]):
        gravgse[i, 0] = x_vals[i, 0]
        gravgse[i, 1] = np.mean(gr_p[:, i])

    # --- Important: Correct g(0) ---
    gravgse[0, 1] = g0_value
    print(f"Corrected g(0) in gravgse = {g0_value}")
    # --- Done correcting ---

    # Save to Excel
    output_filename = os.path.join(data_dir, os.path.splitext(os.path.basename(file_path))[0] + '.xlsx')

    # If file exists, remove it first to ensure overwriting
    if os.path.exists(output_filename):
        try:
            os.remove(output_filename)
            print(f"Old file {output_filename} deleted.")
        except PermissionError:
            print(f"⚠️ Please close {output_filename} and rerun.")
            raise

    df_tot = pd.DataFrame(gr_p[:, :])
    dfavgse = pd.DataFrame(gravgse[:, :])

    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        dfavgse.to_excel(writer, sheet_name="vn-A-2-crop_wfilter-avgstd", header=False, index=False)

    print(f"✅ Processed and saved: {output_filename}")
    return df_tot, dfavgse

# Function to process all TIFFs in a folder
def process_tiff_folder(folder_path, filtersize=256):
    tiff_files = glob.glob(os.path.join(folder_path, '*.tif'))
    for tiff_file_path in tiff_files:
        print(f"Processing {tiff_file_path}...")
        process_tiff_file(tiff_file_path, filtersize=filtersize)

# Start processing
process_tiff_folder(data_dir)


Processing F:\sia-exp1,2-control,blebb,cytd-analyzed tobe plotted\wt-vn-sia-exp1-ctd.blebb,controlzstack-analyzed to be plotted\blebb-vn-sia\vn-smol region-300by 300\vn01031-1.tif...
g(0) from central pixel = 0.9999999999999993
Corrected g(0) in gravgse = 0.9999999999999993
Old file F:\sia-exp1,2-control,blebb,cytd-analyzed tobe plotted\wt-vn-sia-exp1-ctd.blebb,controlzstack-analyzed to be plotted\blebb-vn-sia\vn-smol region-300by 300\vn01031-1.xlsx deleted.
✅ Processed and saved: F:\sia-exp1,2-control,blebb,cytd-analyzed tobe plotted\wt-vn-sia-exp1-ctd.blebb,controlzstack-analyzed to be plotted\blebb-vn-sia\vn-smol region-300by 300\vn01031-1.xlsx
Processing F:\sia-exp1,2-control,blebb,cytd-analyzed tobe plotted\wt-vn-sia-exp1-ctd.blebb,controlzstack-analyzed to be plotted\blebb-vn-sia\vn-smol region-300by 300\vn01031-2.tif...
g(0) from central pixel = 1.0
Corrected g(0) in gravgse = 1.0
Old file F:\sia-exp1,2-control,blebb,cytd-analyzed tobe plotted\wt-vn-sia-exp1-ctd.blebb,controlzst